# 코드 생성기

목표: Frontier 모델을 사용하여 Python 코드로부터 고성능 C++ 코드를 생성합니다


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">알림: C++ 또는 Rust 코드 실행은 선택사항입니다</h2>
            <span style="color:#f71;">대안으로 어제 안내한 웹사이트에서 실행할 수 있습니다</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">중요 참고사항</h1>
            <span style="color:#900;">
            이 실습에서는 GPT 5, Claude 4.5 Sonnet, Gemini 2.5 Pro, Grok 4와 같이 약간 더 높은 가격의 고성능 모델을 사용합니다. 비용은 여전히 낮지만, 비용을 최소화하고 싶으시다면 gpt-5-nano와 같은 저비용 모델을 선택하세요.
            </span>
        </td>
    </tr>
</table>

In [1]:
# imports

import os
import io
import sys
import shutil
import platform
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display

# Rust PATH 자동 추가 (rustup 기본 설치 경로)
_cargo_bin = str(Path.home() / ".cargo" / "bin")
if _cargo_bin not in os.environ.get("PATH", ""):
    os.environ["PATH"] = _cargo_bin + os.pathsep + os.environ.get("PATH", "")
print("rustc:", shutil.which("rustc") or "NOT FOUND - Rust 미설치")
print("g++:", shutil.which("g++") or "NOT FOUND")


rustc: C:\Users\yeop\.cargo\bin\rustc.EXE
g++: C:\TDM-GCC-64\bin\g++.EXE


In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")



OpenAI API Key exists and begins sk-proj-
Anthropic API Key not set (and this is optional)
Google API Key exists and begins AI
Grok API Key not set (and this is optional)
Groq API Key not set (and this is optional)
OpenRouter API Key not set (and this is optional)


In [3]:
# Connect to client libraries

openai = OpenAI()

anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)



In [4]:
# gpt-4o-mini를 맨 앞(기본값)으로 설정 - 빠르고 저렴
models = ["gpt-4o-mini", "gpt-5", "claude-sonnet-4-5-20250929", "grok-4", "gemini-2.5-pro", "qwen2.5-coder", "deepseek-coder-v2", "gpt-oss:20b", "qwen/qwen3-coder-30b-a3b-instruct", "openai/gpt-oss-120b"]

clients = {
    "gpt-4o-mini": openai,
    "gpt-5": openai,
    "claude-sonnet-4-5-20250929": anthropic,
    "grok-4": grok,
    "gemini-2.5-pro": gemini,
    "openai/gpt-oss-120b": groq,
    "qwen2.5-coder": ollama,
    "deepseek-coder-v2": ollama,
    "gpt-oss:20b": ollama,
    "qwen/qwen3-coder-30b-a3b-instruct": openrouter,
}

In [5]:
from system_info import retrieve_system_info, rust_toolchain_info

system_info = retrieve_system_info()
rust_info = rust_toolchain_info()
rust_info

{'installed': True,
 'rustc': {'path': 'C:\\Users\\yeop\\.cargo\\bin\\rustc.EXE',
  'version': 'rustc 1.94.0 (4a4ef493e 2026-03-02)',
  'host_triple': 'x86_64-pc-windows-msvc',
  'release': '1.94.0',
  'commit_hash': '4a4ef493e3a1488c6e321570238084b38948f6db'},
 'cargo': {'path': 'C:\\Users\\yeop\\.cargo\\bin\\cargo.EXE',
  'version': 'cargo 1.94.0 (85eff7c80 2026-01-15)'},
 'rustup': {'path': 'C:\\Users\\yeop\\.cargo\\bin\\rustup.EXE',
  'version': 'rustup 1.29.0 (28d1352db 2026-03-05)',
  'active_toolchain': 'stable-x86_64-pc-windows-msvc (default)',
  'default_toolchain': '',
  'toolchains': ['stable-x86_64-pc-windows-msvc (active, default)'],
  'targets_installed': ['x86_64-pc-windows-msvc']},
 'rust_analyzer': {'path': 'C:\\Users\\yeop\\.cargo\\bin\\rust-analyzer.EXE'},
 'env': {'CARGO_HOME': 'C:\\Users\\yeop\\.cargo',
  'RUSTUP_HOME': 'C:\\Users\\yeop\\.rustup',
  'RUSTFLAGS': '',
  'CARGO_BUILD_TARGET': ''},
 'execution_examples': ['"C:\\Users\\yeop\\.cargo\\bin\\cargo.EXE" buil

In [6]:
message = f"""
Here is a report of the system information for my computer.
I want to run a Rust compiler to compile a single rust file called main.rs and then execute it in the simplest way possible.
Please reply with whether I need to install a Rust toolchain to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile Rust code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.
Have the maximum possible runtime performance in mind; compile time can be slow. Fastest possible runtime performance for this platform is key.
Reply with the commands in markdown.

System information:
{system_info}

Rust toolchain information:
{rust_info}
"""

response = openai.chat.completions.create(model=models[0], messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

Since you already have the Rust toolchain installed, you do not need to install it again. You can compile and execute your Rust code using `rustc`, which is the Rust compiler.

To compile and run Rust code from a Python script, you can use the following commands. The aim is to maximize runtime performance by compiling with optimizations.

### Python code for compilation and execution

Here’s how you can set it up in your Python script:

```python
import subprocess

# Compile the Rust code with optimizations
compile_command = ["C:\\Users\\yeop\\.cargo\\bin\\rustc.EXE", "main.rs", "-o", "main.exe", "--release"]
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)

# Run the compiled executable
run_command = ["main.exe"]
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)

# Return the output of the executed program
print(run_result.stdout)
```

### Explanation of the commands

1. **Compile Command**:
   - `["C:\\Users\\yeop\\.cargo\\bin\\rustc.EXE", "main.rs", "-o", "main.exe", "--release"]`
     - `C:\\Users\\yeop\\.cargo\\bin\\rustc.EXE`: Path to the Rust compiler.
     - `main.rs`: The Rust source file to compile.
     - `-o main.exe`: Specifies the output filename.
     - `--release`: Enables optimizations for maximum runtime performance.

2. **Run Command**:
   - `["main.exe"]`
     - This command executes the compiled executable.

### Important Notes
- Make sure `main.rs` is in the same directory as your Python script or provide the correct path to it.
- The `--release` flag is crucial for ensuring that your Rust binary is optimized for performance, making it faster at runtime despite potentially longer compile times.

## C++의 경우 어제의 명령으로 덮어쓰기, Rust의 경우 새 명령을 사용하세요

또는 어제처럼 웹사이트를 사용하세요:

 https://www.programiz.com/cpp-programming/online-compiler/

In [7]:
import platform, shutil
from pathlib import Path

# 여기서 언어를 선택하세요. "C++" 또는 "Rust"
language = "Rust"
extension = "rs" if language == "Rust" else "cpp"

# Rust PATH가 안 잡혀 있을 경우 ~/.cargo/bin 을 직접 추가
import os
from pathlib import Path
cargo_bin = str(Path.home() / ".cargo" / "bin")
if cargo_bin not in os.environ.get("PATH", ""):
    os.environ["PATH"] = cargo_bin + os.pathsep + os.environ.get("PATH", "")

if language == "C++":
    if platform.system() == "Windows":
        gpp = shutil.which("g++") or r"C:\TDM-GCC-64\bin\g++.exe"
        if not Path(gpp).exists():
            raise RuntimeError("g++ not found. Install TDM-GCC: winget install -e --id JMEubank.TDM-GCC")
        compile_command = [
            gpp, "-O3", "-march=native", "-flto", "-DNDEBUG", "-std=c++20",
            "main.cpp", "-o", "main.exe", "-static-libstdc++", "-static-libgcc",
        ]
        run_command = [".\\main.exe"]
    else:
        cc = shutil.which("clang++") or shutil.which("g++")
        if not cc:
            raise RuntimeError("No C++ compiler found")
        compile_command = [cc, "-O3", "-march=native", "-flto", "-DNDEBUG", "-std=c++20", "main.cpp", "-o", "main"]
        run_command = ["./main"]
else:  # Rust
    rustc = shutil.which("rustc")
    if not rustc:
        raise RuntimeError("rustc not found. Install Rust: https://rustup.rs")
    compile_command = [
        rustc, "main.rs",
        "-C", "opt-level=3",
        "-C", "target-cpu=native",
        "-C", "codegen-units=1",
        "-C", "lto=fat",
        "-C", "panic=abort",
        "-o", "main.exe" if platform.system() == "Windows" else "main",
    ]
    run_command = [".\\main.exe"] if platform.system() == "Windows" else ["./main"]

print("compile_command:", compile_command)
print("run_command:", run_command)


compile_command: ['C:\\Users\\yeop\\.cargo\\bin\\rustc.EXE', 'main.rs', '-C', 'opt-level=3', '-C', 'target-cpu=native', '-C', 'codegen-units=1', '-C', 'lto=fat', '-C', 'panic=abort', '-o', 'main.exe']
run_command: ['.\\main.exe']


## 이제 본격적으로 시작해봅시다

In [8]:
language = "Rust" # or "C++"
extension = "rs" if language == "Rust" else "cpp"

system_prompt = f"""
Your task is to convert Python code into high performance {language} code.
Respond only with {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to {language} with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.{language} and then compiled and executed; the compilation command is:
{compile_command}
Respond only with {language} code.
Python code to port:

```python
{python}
```
"""

In [9]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [10]:
def write_output(code):
    with open(f"main.{extension}", "w", encoding="utf-8") as f:
        f.write(code)

In [11]:
def port(model, python):
    client = clients[model]
    # reasoning_effort는 o1/o3/o4 계열 추론 모델에서만 지원
    reasoning_models = ('o1', 'o3', 'o4')
    use_reasoning = any(model.startswith(m) for m in reasoning_models)
    kwargs = {"timeout": 120}
    if use_reasoning:
        kwargs['reasoning_effort'] = 'high'
    response = client.chat.completions.create(model=model, messages=messages_for(python), **kwargs)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```rust','').replace('```','')
    return reply

In [12]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [13]:
# Use the commands from GPT 5

def compile_and_run(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [14]:
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [15]:
from styles import CSS

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to {language}") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (original)",
                value=python_hard,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            cpp = gr.Code(
                label=f"{language} (generated)",
                value="",
                language="cpp",
                lines=26
            )

    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(models, value=models[0], show_label=False)
        convert = gr.Button(f"Port to {language}", elem_classes=["convert-btn"])
        cpp_run = gr.Button(f"Run {language}", elem_classes=["run-btn", "cpp"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            cpp_out = gr.TextArea(label=f"{language} result", lines=8, elem_classes=["cpp-out"])

    convert.click(fn=port, inputs=[model, python], outputs=[cpp])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    cpp_run.click(fn=compile_and_run, inputs=[cpp], outputs=[cpp_out])

ui.launch(inbrowser=True)


C:\Users\yeop\AppData\Local\Temp\ipykernel_14904\3949588770.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to {language}") as ui:
C:\Users\yeop\AppData\Local\Temp\ipykernel_14904\3949588770.py:3: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to {language}") as ui:


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## RESULTS!

Qwen 2.5 Coder: 실패  
Gemini 2.5 Pro: 실패  
DeepSeek Coder v2: 실패  
Qwen3 Coder 30B: 실패  
Claude Sonnet 4.5: 실패    
GPT-5: 실패    

3위: GPT-oss-20B: 0.000341  
2위: Grok 4: 0.000317  
**1위: OpenAI GPT-OSS 120B: 0.000304**  

In [16]:
print(f"In Ed's experimenet, the GPT-OSS 120B model outcome is {33.755209/0.000304:,.0f} times faster than the Python code.")